# Phase 3 — The 2026 approach: CLIP zero-shot (actually run)

> **This is not a design note.** The modern approach was *run* on the same 1,050
> Flipkart products and measured. Full pipeline: `src/run_clip_zeroshot.py` →
> `results_clip_zeroshot.json`.

In 2023 the hard part was *fusing* two unrelated feature spaces (Word2Vec vs
VGG16) — hence the early-vs-late fusion study in notebook 02. **CLIP removes the
problem by construction:** it is trained so an image and its text land at the
*same place* in one vector space. Classification, retrieval and even zero-shot
labelling then become cosine-similarity lookups.

## 1 · Zero-shot classification — no training at all

With CLIP you classify into the 7 categories **without a single labelled
example**: embed the category *names* as text, embed each product (image and/or
text), and assign the nearest category by cosine similarity. In 2023 this needed
K-Means + manual Hungarian alignment; here it is a dot product.

In [ ]:
from sentence_transformers import SentenceTransformer
clip = SentenceTransformer("clip-ViT-B-32")        # runs on CPU

# CLIP's text encoder is capped at 77 tokens; Flipkart descriptions run to ~700,
# so we force truncation at tokenisation time (see run_clip_zeroshot.py).

prompts = [f"a photo of a product in the category: {c}" for c in cats]
label_emb = clip.encode(prompts, normalize_embeddings=True)       # (7, 512)

img_emb = clip.encode(images, normalize_embeddings=True)          # (n, 512)
txt_emb = clip.encode(descriptions, normalize_embeddings=True)    # (n, 512)

pred_img  = (img_emb @ label_emb.T).argmax(1)                     # zero-shot, image
pred_txt  = (txt_emb @ label_emb.T).argmax(1)                     # zero-shot, text
fused     = img_emb + txt_emb                                     # fusion = a mean,
fused    /= np.linalg.norm(fused, axis=1, keepdims=True)          #   same shared space
pred_fuse = (fused @ label_emb.T).argmax(1)

### Measured output (zero-shot, n=1,050, no training)

```
CLIP zero-shot · text only     acc=0.635  macroF1=0.636
CLIP zero-shot · image only     acc=0.691  macroF1=0.673
CLIP zero-shot · image + text   acc=0.716  macroF1=0.708   <- best zero-shot
```

![CLIP zero-shot vs supervised 2023](../docs/figures/clip_zeroshot.png)
![CLIP zero-shot confusion matrix](../docs/figures/clip_zeroshot_confusion.png)

## 2 · The honest read

| Approach | Training | Accuracy |
|---|:--:|:--:|
| CLIP zero-shot — image + text | **none** | **0.716** |
| Supervised 2023 — VGG16 + SVM | trained | 0.876 |
| Supervised 2023 — late fusion | trained | 0.881 |

Zero-shot CLIP reaches **72%** on 7 classes **with no labelled data** — but stays
~16 points behind the *trained* 2023 pipeline. CLIP does **not** beat a
supervised model here. What it buys is a different cost structure: **no training,
no labels, and multimodal fusion for free** (a mean in the shared space). For a
cold-start catalogue with no labels, 72% out of the box beats collecting and
labelling a training set — that is the real 2026 argument, not "CLIP wins".

## 3 · A real limitation

CLIP's text encoder is hard-capped at **77 tokens**. Flipkart descriptions run up
to ~700 tokens, so the text side is **truncated** and loses information — part of
why text-only zero-shot (0.635) is the weakest of the three. Lifting this
(chunk-and-pool, or a long-context multimodal encoder) is the obvious next step.

## 4 · What I keep from 2023

Not everything should change. The *method* still holds:
- the **fair, shared evaluation protocol** across approaches;
- **fixed seeds** (and `workers=1` for Word2Vec) + a held-out test set;
- **honest negatives** — the small fusion gain in notebook 02, the 77-token cap here.

Modern models change the *tools*, not the *rigour*.